# CyVerse drone campaign readiness — November 2026

Run this notebook in your **CyVerse VM**, from the current repository root. It performs remote inventory and small-file camera/calibration checks; it does **not** download rasters, process flights or upload products.

**The executable VM workflow is [drone_cyverse_production.ipynb](drone_cyverse_production.ipynb).** It connects source-bound calibration, explicitly declared per-pixel camera geometry, exclusion masks, translation delivery controls, QA and verified upload. Complete actual calibration/source/geometry declarations and review a one-flight pilot before full execution. The old root cloud notebook preserves its historical scalar-view recipe; use the new executable notebook instead.

This notebook gives a per-flight preparation list before committing VM disk and compute. Original Data Store files remain unchanged. Run the cells in order; no local Mac paths are required.


## 1. VM setup and configuration

Use a persistent VM disk. Install this checkout into the notebook kernel with `python -m pip install -e '.[tests,analytics,landsat]'`, restart the kernel, and authenticate GoCommands with `gocmd init` in a terminal. Do not store credentials in the notebook. Copying this notebook alone is insufficient: the matching repository/scripts are required.

Camera tables live in the original `summer-2023` collection, not necessarily alongside the `summer-2023-10cm-10k` raster derivatives. Configure 2024 independently; its camera coverage has not been established by the three-site 2023 work. Exact ExportPackage names associate collections; spelling differences **inside** a package are allowed, ambiguous matches are rejected.


In [ ]:
from pathlib import Path
import json
import sys

REPO = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / 'src/spectralbridge').is_dir() and (path / 'pyproject.toml').is_file()
)
WORK = Path('/home/jovyan/data-store/spectralbridge_drone/readiness_20261120_v1')
REMOTE_PARENT = 'i:/iplant/home/shared/earthlab/macrosystems/field-data/output'
YEARS = (2023, 2024)
RASTER_SOURCES = {year: f'{REMOTE_PARENT}/summer-{year}-10cm-10k' for year in YEARS}
CAMERA_SOURCES = {year: f'{REMOTE_PARENT}/summer-{year}' for year in YEARS}
MANIFEST = REPO / 'src/spectralbridge/data/drone_field_manifest.csv'
GOCMD = 'gocmd'
# A reviewed local JSON keyed by exact remote raster ExportPackage path.
# See section 4 for the per-flight contract. Missing files are reported, not guessed.
FLIGHT_CONTRACTS = WORK / 'flight_contracts.json'
TRANSLATION_REGISTRY = None  # Set to a reviewed physical-unit registry in the VM.
REMOTE_READS = False  # Enable after checking the collections above.


In [ ]:
import inspect
import subprocess
import spectralbridge.pipelines.drone as drone

assert (REPO / 'scripts/drone_vm_readiness.py').is_file(), 'Use the matching repository checkout.'
assert Path(drone.__file__).resolve().is_relative_to(REPO / 'src'), 'Restart kernel after editable installation.'
sys.path.insert(0, str(REPO / 'scripts'))
from drone_cloud_workflow import JsonGocmdBackend, write_json
from drone_vm_readiness import (
    camera_entry, stage_camera, associated_camera_collection,
    compose_brightness_calibration, readiness_report,
)
WORK.mkdir(parents=True, exist_ok=True)
print('Producer:', drone.__file__)
print('Revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())
print('Declared camera geometry parameter present:', 'camera_geometry' in inspect.signature(drone.run_drone_pipeline).parameters)
print('Brightness JSON is separate from the AOP-derived translation registry.')


## 2. Inventory the complete manifest population

This lists remote collections without transferring raster data. Missing manifest flights and missing years remain blockers. A permission/listing error remains an error; it is not evidence that a collection is empty.


In [ ]:
from spectralbridge import inspect_drone_collection

assert REMOTE_READS, 'Review configuration, enable REMOTE_READS, then run inventory.'
backend = JsonGocmdBackend(GOCMD)
backend.ensure_available()
inventory = inspect_drone_collection(
    RASTER_SOURCES, years=YEARS, manifest=MANIFEST,
    backend=backend, inventory_output_dir=WORK / 'inventory',
)
print(json.dumps(inventory.year_summaries(), indent=2, default=str))
print('Eligible packages:', len(inventory.eligible_packages))
print('Excluded:', [(p.package_name, p.reason) for p in inventory.excluded_packages])


## 3. Fetch and validate the small camera tables

The reader checks Metashape row counts, photo IDs, rotations and projected CRS, retaining the original hash and declared CRS. This does not establish mosaic photo contributors, pose/vertical conventions or photo clock/timezone. Goldhill's historical CRS discrepancy needs an explicit evidence-backed interpretation; it is never automatically relabelled.

`camera_overrides` in a reviewed flight contract may contain `crs_override` and `crs_override_provenance`. Retain independent supporting evidence. Errors below become per-flight preparation items, while other flights are inventoried.


In [ ]:
contracts = json.loads(FLIGHT_CONTRACTS.read_text()) if FLIGHT_CONTRACTS.is_file() else {}
cameras, camera_errors = {}, {}
for number, package in enumerate(inventory.eligible_packages, 1):
    key = package.remote_package_path
    print(f'[{number}/{len(inventory.eligible_packages)}] camera: {package.package_name}', flush=True)
    try:
        entry = camera_entry(backend, associated_camera_collection(package, CAMERA_SOURCES))
        camera_contract = contracts.get(key, {}).get('camera_overrides', {})
        cameras[key] = stage_camera(
            backend, entry, WORK / 'cameras' / str(package.year) / package.package_name,
            **camera_contract,
        )
    except Exception as exc:
        camera_errors[key] = f'{type(exc).__name__}: {exc}'
write_json(WORK / 'camera_audit.json', {'valid': cameras, 'errors': camera_errors})
print(json.dumps(camera_errors, indent=2))


## 4. Bind the incoming brightness JSON to source identity and wavelengths

The supplied coefficients are **per-band drone calibration**, not Landsat translation. The proposed adapter contract is:

`decoded_reflectance = source_scale * stored_value + source_offset`

`calibrated_reflectance = gain * decoded_reflectance + offset`

Offsets are in fractional reflectance. Combined coefficients are `gain * source_scale` and `gain * source_offset + offset`, passed through the existing affine radiometry model. No numeric-range inference or second sensor calibration is performed. Confirm this direction/units with the coefficient provider before adapting their JSON. Positive finite gains, finite offsets, complete wavelength order, source SHA-256, flight identity and provenance are required. No coefficients are invented here.

A `flight_contracts.json` entry, keyed by the exact raster ExportPackage remote path, supplies `flight_id`, `source_sha256`, `wavelengths_nm`, `source_radiometry` (the existing reviewed decoding record), and `brightness_json` (VM path). The brightness file uses `schema_version: 1`, `status: "REVIEWED"`, `calibration_id`, `flight_id`, `source_sha256`, `provenance`, `equation: "calibrated_reflectance = gain * decoded_reflectance + offset"`, and `bands: [{"wavelength_nm": ..., "gain": ..., "offset": ...}, ...]`.

These checks validate supplied declarations. The executable production notebook hashes actual downloaded sources, binds provider declarations to exact retained-value derivatives and includes calibration contents in checkpoints. This preparation notebook does not apply the composed record to any raster.


In [ ]:
calibrations, calibration_errors, wavelengths = {}, {}, {}
for package in inventory.eligible_packages:
    key = package.remote_package_path
    try:
        contract = contracts[key]
        wavelengths[key] = contract['wavelengths_nm']
        coefficients = json.loads(Path(contract['brightness_json']).read_text())
        calibrations[key] = compose_brightness_calibration(
            coefficients, flight_id=contract['flight_id'], source_sha256=contract['source_sha256'],
            wavelengths_nm=contract['wavelengths_nm'], source_radiometry=contract['source_radiometry'],
        )
    except Exception as exc:
        calibration_errors[key] = f'{type(exc).__name__}: {exc}'
write_json(WORK / 'calibration_audit.json', {'composed_records': calibrations, 'errors': calibration_errors})
print(json.dumps(calibration_errors, indent=2))


## 5. Save the readiness report — no processing or upload

A complete per-flight declaration is not a production completion certificate. The report checks whether the installed producer has the execution integration and whether the selected flight contracts are supplied. A complete preparation report can be **READY_FOR_PILOT**; it does not assert production completion or approve scientific assumptions. The supplied translation registry is checked independently for each wavelength configuration; the historical packaged registry may fail its physical-unit checks.


In [ ]:
report = readiness_report(
    inventory, cameras=cameras, calibrations=calibrations,
    translation_coefficients=TRANSLATION_REGISTRY, wavelengths_by_flight=wavelengths, flight_contracts=contracts,
)
report.update(camera_errors=camera_errors, calibration_errors=calibration_errors)
write_json(WORK / 'readiness_report.json', report)
print(json.dumps(report, indent=2, default=str))
print('Saved:', WORK / 'readiness_report.json')


## Proceed to the executable VM workflow

Open [drone_cyverse_production.ipynb](drone_cyverse_production.ipynb) for source staging, an explicit per-flight contract template, pilot processing, full-campaign processing and verified uploads. The public drone pipeline now accepts `brightness_calibration`, `camera_geometry`, `processing_mask` and explicit translation target/naming/storage controls.

Remaining field preparation is scientific input review: actual provider coefficients and gain/offset units; source encoding/identity; same-grid surface with documented height datum; camera assignment and CRS/clock evidence; chosen correction/mask policy; complete camera coverage across the manifest. A nearest-camera scenario stays labelled as assumed contributors. No 10% threshold or experimental correction branch becomes a universal default.

The executable notebook requires a successful reviewed pilot before the full campaign. Individual unsupported BRDF models remain blocked. Upload completion follows full payload SHA-256 readback. This preparation notebook still has no processing/upload escape hatch; keep the historical root notebook for provenance.
